# Pilot: modern AABB quatrain generation on an L40

A small test run (default 300 quatrains) to check **quality, pass rate and speed** before generating the full 20–50k.

What it does:
1. Builds a bank of rhyme pairs from common modern English words (CMUdict + word frequency)
2. Builds varied prompts (topic × tone × voice × rhyme words)
3. Generates with a local model through vLLM
4. Checks each quatrain automatically: 4 lines, AABB rhyme, requested end words, syllable range, unknown-word rate, duplicates
5. Shows stats, good and bad samples, and a time estimate for the full run

Half the prompts give the model fixed end words and half let it choose, so you can see which mode works better.

## 1. Install

In [5]:
%pip install -q vllm pronouncing wordfreq pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Settings

The default model is a fast mixture-of-experts model that fits on the L40 in FP8. Swap `MODEL_ID` to compare others in the pilot.

In [6]:
import os, re, json, time, random
from collections import Counter

MODEL_ID = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"
# Alternatives to compare:
# MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"
# MODEL_ID = "google/gemma-3-27b-it"   # needs a Hugging Face login and licence acceptance

N_SAMPLES        = 300      # pilot size
GIVEN_RHYME_FRAC = 0.5      # share of prompts with fixed end words; the rest let the model choose
TEMPERATURE      = 0.9
TOP_P            = 0.95
MAX_TOKENS       = 120      # a quatrain is about 50-70 tokens
SEED             = 42

SYL_MIN, SYL_MAX = 6, 12    # allowed syllables per line
PAIR_SYL_DIFF    = 3        # max syllable difference within a rhyming pair
MAX_OOV_RATE     = 0.15     # max share of words not in CMUdict per quatrain

OUT_DIR = "pilot_out"
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)

## 3. Phonetic helpers

In [7]:
import pronouncing

WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def words_of(line):
    return [w.lower() for w in WORD_RE.findall(line)]

def last_word(line):
    w = words_of(line)
    return w[-1] if w else ""

def all_phones(word):
    return pronouncing.phones_for_word(word.lower())

def rhyme_parts(word):
    return {pronouncing.rhyming_part(p) for p in all_phones(word)}

def rhymes(a, b):
    # True if any pronunciation of a rhymes with any pronunciation of b (and they're different words)
    a, b = a.lower(), b.lower()
    if not a or not b or a == b:
        return False
    if a.endswith(b) or b.endswith(a):          # "play" / "replay" is not a real rhyme
        return False
    ra, rb = rhyme_parts(a), rhyme_parts(b)
    return bool(ra and rb and ra & rb)

def word_syllables(word):
    ph = all_phones(word)
    if ph:
        return pronouncing.syllable_count(ph[0])
    # fallback for unknown words: count vowel groups
    groups = re.findall(r"[aeiouy]+", word.lower())
    n = len(groups)
    if word.lower().endswith("e") and n > 1 and not word.lower().endswith("le"):
        n -= 1
    return max(n, 1)

def line_syllables(line):
    return sum(word_syllables(w) for w in words_of(line))

def oov_rate(lines):
    ws = [w for l in lines for w in words_of(l)]
    if not ws:
        return 1.0
    return sum(1 for w in ws if not all_phones(w)) / len(ws)

# quick sanity checks
assert rhymes("late", "wait") and rhymes("phone", "alone")
assert not rhymes("late", "phone") and not rhymes("play", "replay")
print("syllables:", line_syllables("I missed the bus and now I'm running late"))

syllables: 10


## 4. Rhyme-pair bank

Common words only, so the quatrains stay modern and everyday. Each rhyme sound is used once per prompt pair, and the A and B sounds always differ.

In [8]:
from wordfreq import top_n_list

STOP = set('''a an the and or but if of to in on at by for with from as is are was were be been am
i you he she it we they me him her us them my your his its our their this that these those
not no so do does did has have had will would can could shall should may might must
there here what which who whom whose when where why how than then just also very'''.split())

common = [w for w in top_n_list("en", 15000)
          if w.isalpha() and len(w) >= 3 and w not in STOP and all_phones(w)]

by_sound = {}
for w in common:
    rp = pronouncing.rhyming_part(all_phones(w)[0])
    by_sound.setdefault(rp, []).append(w)

pairs_by_sound = {}
for rp, ws in by_sound.items():
    ps = [(a, b) for i, a in enumerate(ws) for b in ws[i+1:] if rhymes(a, b)]
    if ps:
        pairs_by_sound[rp] = ps[:40]          # cap per sound so huge families don't dominate

sounds = list(pairs_by_sound)
print(f"{len(common)} common words, {len(sounds)} rhyme sounds with usable pairs")
print("examples:", [random.choice(pairs_by_sound[s]) for s in random.sample(sounds, 8)])

def sample_rhyme_plan():
    s1, s2 = random.sample(sounds, 2)          # two different rhyme sounds
    return random.choice(pairs_by_sound[s1]), random.choice(pairs_by_sound[s2])

14166 common words, 1782 rhyme sounds with usable pairs
examples: [('nigerian', 'presbyterian'), ('gonna', 'marijuana'), ('cook', 'brook'), ('spaghetti', 'sweaty'), ('hurry', 'furry'), ('highly', 'riley'), ('teachers', 'creatures'), ('libraries', 'tories')]


## 5. Prompt seeds

Random topic × tone × voice combinations keep the output varied. For the full run, extend `TOPICS` to a few hundred.

In [9]:
TOPICS = [
    "missing the morning bus", "a group project where nobody helps", "the last slice of pizza",
    "a phone at one percent battery", "Monday mornings", "a cat that ignores you", "rainy weekends",
    "a gym membership you never use", "waiting for exam results", "a noisy neighbour",
    "online shopping regrets", "a houseplant that keeps dying", "the first day at a new job",
    "a friend who is always late", "burnt toast", "a long queue at the coffee shop",
    "forgetting someone's name", "the summer heat", "a road trip with bad directions",
    "a video call with a frozen screen", "a messy bedroom", "learning to cook", "a broken umbrella",
    "staying up too late", "a lost set of keys", "a birthday nobody remembered", "a cheap haircut",
    "sending a text to the wrong person", "the alarm clock", "a crowded train", "homework due tomorrow",
    "a dog that steals socks", "a failed diet", "an awkward first date", "the office printer",
    "a slow wifi connection", "grocery shopping while hungry", "a song stuck in your head",
    "moving to a new city", "a rainy football match", "a talkative taxi driver", "a group chat",
    "trying to wake up early", "a surprise party", "a spilled drink", "the weekend ending too soon",
    "a long flight", "a boring meeting", "a forgotten password", "the smell of fresh bread",
]
TONES  = ["funny", "sarcastic", "wistful", "hopeful", "deadpan", "dramatic", "playful", "grumpy"]
VOICES = ["first person (I)", "second person (you)", "a narrator describing someone"]

SYSTEM = (
    "You write short modern poems. Output ONLY the poem: exactly 4 lines, "
    "no title, no numbering, no quotation marks, no blank lines, no explanation."
)

def build_prompt(topic, tone, voice, plan):
    rules = (
        "Write one quatrain with rhyme scheme AABB: lines 1 and 2 rhyme with each other, "
        "lines 3 and 4 rhyme with each other, using a different rhyme sound.\n"
        "Each line should be 7 to 11 syllables.\n"
        "Use modern everyday English. No archaic words (thee, thou, doth), no cliches, simple punctuation.\n"
        f"Topic: {topic}\nTone: {tone}\nVoice: {voice}\n"
    )
    if plan:
        (a1, a2), (b1, b2) = plan
        rules += (f"Line 1 must end with the word '{a1}', line 2 with '{a2}', "
                  f"line 3 with '{b1}', line 4 with '{b2}'.\n")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": rules}]

jobs = []
for i in range(N_SAMPLES):
    plan = sample_rhyme_plan() if i < N_SAMPLES * GIVEN_RHYME_FRAC else None
    job = {"id": i, "topic": random.choice(TOPICS), "tone": random.choice(TONES),
           "voice": random.choice(VOICES), "plan": plan}
    job["messages"] = build_prompt(job["topic"], job["tone"], job["voice"], plan)
    jobs.append(job)

print(jobs[0]["messages"][1]["content"])

Write one quatrain with rhyme scheme AABB: lines 1 and 2 rhyme with each other, lines 3 and 4 rhyme with each other, using a different rhyme sound.
Each line should be 7 to 11 syllables.
Use modern everyday English. No archaic words (thee, thou, doth), no cliches, simple punctuation.
Topic: burnt toast
Tone: funny
Voice: a narrator describing someone
Line 1 must end with the word 'own', line 2 with 'tone', line 3 with 'find', line 4 with 'resigned'.



## 6. Load the model

In [10]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    max_model_len=2048,           # prompts are short; a small context leaves more memory for batching
    gpu_memory_utilization=0.90,
    seed=SEED,
)
params = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS,
                        repetition_penalty=1.05)

/common/home/projectgrps/CS480/CS480G3/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(MainProcess pid=3196254) INFO 10-19 19:33:57 [api_utils.py:286] non-default args: {'seed': 42, 'max_model_len': 2048, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507-FP8'}
(MainProcess pid=3196254) INFO 10-19 19:34:06 [model.py:733] Resolved architecture: Qwen3MoeForCausalLM
(MainProcess pid=3196254) INFO 10-19 19:34:06 [model.py:2152] Using max model len 2048
(MainProcess pid=3196254) INFO 10-19 19:34:07 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=3196254) INFO 10-19 19:34:07 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=3196254) INFO 10-19 19:34:07 [compilation.py:317] Enabled custom fusions: norm_quant, act_quant


Fetching 10 files: 100%|██████████| 10/10 [00:02<00:00,  3.71it/s]


(EngineCore pid=3197804) INFO 10-19 19:34:11 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', speculative_config=None, tokenizer='Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=fp8, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:02<00:08,  2.95s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:05<00:05,  2.99s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:08<00:02,  2.99s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:09<00:00,  2.09s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:09<00:00,  2.42s/it]



(EngineCore pid=3197804) INFO 10-19 19:40:11 [default_loader.py:501] Loading weights took 9.70 seconds
(EngineCore pid=3197804) WARNING 10-19 19:40:12 [marlin_utils_fp8.py:108] Your GPU does not have native support for FP8 computation but FP8 quantization is being used. Weight-only FP8 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.
(EngineCore pid=3197804) INFO 10-19 19:40:12 [fp8.py:856] Using MoEPrepareAndFinalizeNoDPEPModular
(EngineCore pid=3197804) INFO 10-19 19:40:13 [model_runner.py:424] Model loading took 29.11 GiB memory and 360.972063 seconds
(EngineCore pid=3197804) INFO 10-19 19:40:13 [topk_topp_sampler.py:124] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=3197804) INFO 10-19 19:40:13 [torch_utils.py:295] Reducing Torch threads from 4 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=3197804) INFO 10-19 19:40:13 [utils.py:347] Using LBNHC KV cache la

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00, 17.50it/s]


(EngineCore pid=3197804) INFO 10-19 19:41:11 [model_runner.py:1127] Graph capturing finished in 10 secs, took 0.59 GiB
(EngineCore pid=3197804) INFO 10-19 19:41:11 [gpu_worker.py:759] Available KV cache memory: 8.95 GiB
(EngineCore pid=3197804) INFO 10-19 19:41:11 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.8857 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9143. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=3197804) INFO 10-19 19:41:11 [kv_cache_utils.py:2327] GPU KV cache size: 97,760 tokens, Maximum concurrency for 2,048 tokens per request: 47.73x
(EngineCore pid=3197804) INFO 10-19 19:41:12 [kernel_warmup.py:176] JIT kernel warmup starting.
(EngineCore pid=3197804) INFO 10-19 19:41:12 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s

2026-10-19 19:41:17,229 - INFO - core.py:564 - flashinfer.jit: Building JIT module sampling; this can take several minutes on first use.
Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:02<00:00, 16.99it/s]


(EngineCore pid=3197804) INFO 10-19 19:42:16 [model_runner.py:1127] Graph capturing finished in 7 secs, took 0.33 GiB
(EngineCore pid=3197804) INFO 10-19 19:42:16 [gpu_worker.py:971] CUDA graph pool memory: 0.33 GiB (actual), 0.64 GiB (estimated), difference: 0.31 GiB (92.9%).
(EngineCore pid=3197804) INFO 10-19 19:42:16 [gpu_worker.py:1034] Free memory on device (43.97/44.39 GiB) on startup. Desired GPU memory utilization is (0.9, 39.95 GiB). Actual usage is 30.0 GiB for consumed memory (weights + non-torch), 1.0 GiB for peak activation, and 0.33 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=9099065037` (8.47 GiB) to fit into requested memory, or `--kv-cache-memory=13410494976` (12.49 GiB) to fully utilize gpu memory. Current kv cache memory in use is 8.95 GiB.
(EngineCore pid=3197804) INFO 10-19 19:42:18 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=3197804) I

## 7. Generate

In [11]:
t0 = time.time()
outputs = llm.chat([j["messages"] for j in jobs], params, use_tqdm=True)
elapsed = time.time() - t0

out_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
for j, o in zip(jobs, outputs):
    j["raw"] = o.outputs[0].text
print(f"{len(jobs)} quatrains in {elapsed:.1f}s | {out_tokens} output tokens | "
      f"{out_tokens/elapsed:.0f} tok/s | {len(jobs)/elapsed:.1f} quatrains/s")

Processed prompts: 100%|██████████| 300/300 [00:04<00:00, 69.63it/s, est. speed input: 11027.37 toks/s, output: 3105.94 toks/s] 

300 quatrains in 4.4s | 13381 output tokens | 3011 tok/s | 67.5 quatrains/s


## 8. Parse and check

In [12]:
def parse(raw):
    lines = []
    for l in raw.strip().splitlines():
        l = l.strip().strip('"').strip("“”").strip()
        l = re.sub(r"^(\d+[\.\):]|[-*•])\s*", "", l)        # drop numbering / bullets
        if l and not l.lower().startswith(("here", "title", "sure")):
            lines.append(l)
    return lines

def check(job):
    lines = parse(job["raw"])
    r = {"lines": lines, "reasons": []}
    if len(lines) != 4:
        r["reasons"].append(f"line_count={len(lines)}")
        r["pass"] = False
        return r
    ends = [last_word(l) for l in lines]
    r["ends"] = ends
    if not rhymes(ends[0], ends[1]): r["reasons"].append("A_no_rhyme")
    if not rhymes(ends[2], ends[3]): r["reasons"].append("B_no_rhyme")
    if rhymes(ends[0], ends[2]):     r["reasons"].append("A_equals_B")
    if job["plan"]:
        want = [w for pair in job["plan"] for w in pair]
        if ends != want: r["reasons"].append("wrong_end_words")
    syl = [line_syllables(l) for l in lines]
    r["syllables"] = syl
    if any(s < SYL_MIN or s > SYL_MAX for s in syl):      r["reasons"].append("syllable_range")
    if abs(syl[0]-syl[1]) > PAIR_SYL_DIFF or abs(syl[2]-syl[3]) > PAIR_SYL_DIFF:
        r["reasons"].append("pair_unbalanced")
    r["oov"] = round(oov_rate(lines), 3)
    if r["oov"] > MAX_OOV_RATE:                           r["reasons"].append("oov")
    r["pass"] = not r["reasons"]
    return r

seen = set()
for j in jobs:
    j.update(check(j))
    key = " / ".join(l.lower() for l in j["lines"])
    if j["pass"] and key in seen:
        j["pass"] = False; j["reasons"].append("duplicate")
    seen.add(key)

passed = [j for j in jobs if j["pass"]]
print(f"Passed: {len(passed)}/{len(jobs)} ({len(passed)/len(jobs):.0%})")

Passed: 17/300 (6%)


## 9. Stats

In [13]:
import pandas as pd

df = pd.DataFrame([{
    "mode": "given_rhymes" if j["plan"] else "free_rhymes",
    "pass": j["pass"], "reasons": ",".join(j["reasons"]),
    "oov": j.get("oov"), "syllables": j.get("syllables"),
} for j in jobs])

print("Pass rate by mode:")
print(df.groupby("mode")["pass"].mean().map("{:.0%}".format), "\n")

print("Failure reasons (a quatrain can fail for several):")
print(Counter(r for j in jobs for r in j["reasons"]).most_common(), "\n")

all_syl = [s for j in jobs if j.get("syllables") for s in j["syllables"]]
if all_syl:
    print(f"Syllables per line: mean {sum(all_syl)/len(all_syl):.1f}, min {min(all_syl)}, max {max(all_syl)}")

end_pairs = Counter(tuple(sorted(j["ends"][k:k+2])) for j in passed for k in (0, 2))
print(f"Distinct rhyme pairs among passed: {len(end_pairs)}; most common:", end_pairs.most_common(8))

all_words = [w for j in passed for l in j["lines"] for w in words_of(l)]
if all_words:
    print(f"Vocabulary: {len(set(all_words))} distinct words in {len(all_words)} tokens "
          f"(type/token ratio {len(set(all_words))/len(all_words):.2f})")

Pass rate by mode:
mode
free_rhymes     4%
given_rhymes    7%
Name: pass, dtype: str 

Failure reasons (a quatrain can fail for several):
[('A_no_rhyme', 240), ('B_no_rhyme', 233), ('wrong_end_words', 130), ('syllable_range', 59), ('A_equals_B', 51), ('pair_unbalanced', 23), ('line_count=3', 2)] 

Syllables per line: mean 10.3, min 6, max 21
Distinct rhyme pairs among passed: 33; most common: [(('frame', 'name'), 2), (('societies', 'varieties'), 1), (('prestigious', 'religious'), 1), (('freaked', 'leaked'), 1), (('casting', 'fasting'), 1), (('anne', 'man'), 1), (('lemon', 'yemen'), 1), (('compartment', 'department'), 1)]
Vocabulary: 296 distinct words in 522 tokens (type/token ratio 0.57)


## 10. Samples

In [14]:
def show(j):
    tag = "given " + str(j["plan"]) if j["plan"] else "free"
    print(f"[{j['id']}] {j['topic']} | {j['tone']} | {tag}")
    for l, s in zip(j["lines"], j.get("syllables") or [None]*len(j["lines"])):
        print(f"   {l}" + (f"   ({s})" if s else ""))
    if j["reasons"]: print("   FAIL:", ", ".join(j["reasons"]))
    print()

print("=========== PASSED (random 12) ===========\n")
for j in random.sample(passed, min(12, len(passed))): show(j)

failed = [j for j in jobs if not j["pass"]]
print("=========== FAILED (random 8) ===========\n")
for j in random.sample(failed, min(8, len(failed))): show(j)

=========== PASSED (random 12) ===========

[131] online shopping regrets | funny | given (('meals', 'ideals'), ('election', 'connection'))
   You bought five knitted scarves for winter meals,   (10)
   now your closet’s full of yarny ideals.   (11)
   That midnight snack dip? Not much of an election,   (12)
   and your Wi-Fi’s lost the connection.   (10)

[147] a forgotten password | grumpy | given (('risk', 'disk'), ('productions', 'reductions'))
   He typed it wrong again, that cursed risk,   (9)
   Forgot the old login on his disk.   (9)
   His past attempts feel like bad productions,   (10)
   All hope of access now just reductions.   (10)

[103] a lost set of keys | playful | given (('abuse', 'excuse'), ('duty', 'booty'))
   They blamed the cat for key abuse,   (8)
   But really just a clumsy excuse.   (9)
   Now I’m stuck doing kitchen duty,   (10)
   With nothing but my phone and booty.   (9)

[9] the office printer | hopeful | given (('societies', 'varieties'), ('religious', '

## 11. Time estimate for the full run

In [15]:
pass_rate = len(passed) / len(jobs)
for target in (20_000, 50_000):
    needed = target / max(pass_rate, 1e-6)
    est_min = needed / (len(jobs) / elapsed) / 60
    print(f"{target:,} usable -> generate about {needed:,.0f} (pass rate {pass_rate:.0%}) "
          f"-> roughly {est_min:.0f} min at pilot speed")
print("\nLarger batches usually run faster per quatrain than this pilot, so treat this as an upper bound.")

20,000 usable -> generate about 352,941 (pass rate 6%) -> roughly 87 min at pilot speed
50,000 usable -> generate about 882,353 (pass rate 6%) -> roughly 218 min at pilot speed

Larger batches usually run faster per quatrain than this pilot, so treat this as an upper bound.


## 12. Save

In [16]:
def record(j):
    return {"id": j["id"], "topic": j["topic"], "tone": j["tone"], "voice": j["voice"],
            "plan": j["plan"], "lines": j["lines"], "ends": j.get("ends"),
            "syllables": j.get("syllables"), "oov": j.get("oov"),
            "pass": j["pass"], "reasons": j["reasons"], "raw": j["raw"]}

with open(f"{OUT_DIR}/pilot_all.jsonl", "w") as f:
    for j in jobs: f.write(json.dumps(record(j)) + "\n")
with open(f"{OUT_DIR}/pilot_passed.jsonl", "w") as f:
    for j in passed: f.write(json.dumps(record(j)) + "\n")

meta = {"model": MODEL_ID, "n": len(jobs), "passed": len(passed), "seconds": round(elapsed, 1),
        "output_tokens": out_tokens, "temperature": TEMPERATURE, "top_p": TOP_P}
json.dump(meta, open(f"{OUT_DIR}/pilot_meta.json", "w"), indent=2)
print("Saved to", OUT_DIR, meta)

Saved to pilot_out {'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', 'n': 300, 'passed': 17, 'seconds': 4.4, 'output_tokens': 13381, 'temperature': 0.9, 'top_p': 0.95}


## What to look at

- **Pass rate by mode**: if given end words pass much more often and still read naturally, use that mode for the full run. If they read forced, use free rhymes and rely on the filters.
- **Failure reasons**: lots of `wrong_end_words` means the model ignores the constraint (try another model or lower temperature); lots of `A_no_rhyme` / `B_no_rhyme` in free mode means it rhymes loosely.
- **Read the passed samples**: the filters only check form. Judge naturalness and humour yourself, and check for cliches.
- **Distinct rhyme pairs and vocabulary**: low numbers mean repetitive output; add topics and tones before scaling up.
- **Compare models**: change `MODEL_ID`, restart the kernel, rerun, and compare pass rate, speed and how the samples read.